In [ ]:
import os
import glob
import re
import numpy as np
import pandas as pd
from datetime import datetime

CONFIG = {
    "INPUT_ROOT": "/kaggle/input/household-ideal",
    "METADATA_DIR": "/kaggle/input/household-ideal/metadata_and_surveys/metadata",
    "SENSOR_DIR": "/kaggle/input/household-ideal/household_sensors/sensordata",
    "WEATHER_FEED": "/kaggle/input/household-ideal/metadata_and_surveys/metadata/weatherfeed.csv",
    "WEATHER_READING": "/kaggle/input/household-ideal/household_sensors/sensordata/weatherreading.csv",
    "OUTPUT_DIR": "/kaggle/working/processed_data",
    
    "NUM_AGENTS": 50,          
    "FREQ": "15min", 
    "REQUIRED_DAYS": 250,     
    
    "FEATURES": [
        "e_kwh", "unit_charge_pence_per_kwh",
        "income_band", "urban_rural_class", "temperature",
        # "tod_sin", "tod_cos", "day_of_week_sin", "day_of_week_cos",
        # "e_kwh_lag1", "e_kwh_roll_mean_4"
    ]
}


In [ ]:
os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)

class IdealDataProcessor:
    def __init__(self, cfg):
        self.cfg = cfg
        self.home_metadata = None
        self.weather_data = None
        self.all_candidates = [] 

    def load_metadata(self):
        print("Loading metadata...")
        meta_dir = self.cfg["METADATA_DIR"]
        
        # 1. Load Home Info
        home_path = os.path.join(meta_dir, "home.csv")
        if os.path.exists(home_path):
            home_df = pd.read_csv(home_path)
            home_df.columns = [c.strip().lower().replace(" ", "_") for c in home_df.columns]
            
            cols_to_keep = ['homeid']
            # Income
            if 'income_band' in home_df.columns:
                home_df['income_band'] = home_df['income_band'].astype('category').cat.codes
                cols_to_keep.append('income_band')
            elif 'equivalised_income_band' in home_df.columns:
                 home_df['income_band'] = home_df['equivalised_income_band'].astype('category').cat.codes
                 cols_to_keep.append('income_band')
            
            # Urban/Rural
            if 'urban_rural_class' in home_df.columns:
                cols_to_keep.append('urban_rural_class')
            
            self.home_metadata = home_df[cols_to_keep].copy()
        
        # 2. Load Tariffs
        tariff_path = os.path.join(meta_dir, "tariff.csv")
        if os.path.exists(tariff_path):
            tar_df = pd.read_csv(tariff_path)
            tar_df.columns = [c.strip().lower().replace(" ", "_") for c in tar_df.columns]
            
            if 'energytype' in tar_df.columns:
                tar_df = tar_df[tar_df['energytype'].str.contains('elec', case=False, na=False)]
            
            if 'notification_date' in tar_df.columns:
                tar_df['notification_date'] = pd.to_datetime(tar_df['notification_date'], errors='coerce')
                tar_df = tar_df.sort_values(['homeid', 'notification_date'])
            
            # Get latest price per home
            tar_df = tar_df.groupby('homeid').tail(1)[['homeid', 'unit_charge_pence_per_kwh']]
            
            # Merge
            if self.home_metadata is not None:
                self.home_metadata = self.home_metadata.merge(tar_df, on='homeid', how='left')
            else:
                self.home_metadata = tar_df

    def load_weather(self):
        feed_path = self.cfg["WEATHER_FEED"]
        read_path = self.cfg["WEATHER_READING"]
        
        if not os.path.exists(feed_path) or not os.path.exists(read_path):
            print("  [!] Weather files not found. Skipping.")
            return

        print("Loading weather...")
        feeds = pd.read_csv(feed_path)
        feeds.columns = [c.strip().lower().replace("_", "") for c in feeds.columns]
        
        if 'weathertype' in feeds.columns:
            temp_feeds = feeds[feeds['weathertype'].str.contains('temp', case=False, na=False)]
        else:
            temp_feeds = feeds[feeds['type'].str.contains('temp', case=False, na=False)]
            
        target_ids = temp_feeds['feedid'].tolist()
        
        readings = pd.read_csv(read_path, usecols=['feedid', 'time', 'value'])
        readings = readings[readings['feedid'].isin(target_ids)].copy()
        readings['value'] = pd.to_numeric(readings['value'], errors='coerce')
        readings = readings.dropna(subset=['value'])
        
        readings['ts'] = pd.to_datetime(readings['time'], utc=True)
        w_df = readings.groupby('ts')['value'].mean()
        w_df = w_df.resample(self.cfg['FREQ']).mean().ffill().reset_index()
        
        self.weather_data = w_df.rename(columns={'value': 'temperature'})[['ts', 'temperature']]

    def find_all_candidates(self):
        """Scans ALL files in the directory and sorts by size (proxy for duration)."""
        print("Scanning ALL files to build candidate pool...")
        files = glob.glob(os.path.join(self.cfg["SENSOR_DIR"], "*.csv"))
        pat = re.compile(r".*/home(?P<homeid>\d+)_.*_electric-mains_electric-combined\.csv$")
        
        candidates = []
        for f in files:
            m = pat.match(f)
            if m:
                size_mb = os.path.getsize(f) / (1024 * 1024)
                candidates.append({
                    "homeid": int(m.group("homeid")),
                    "path": f,
                    "size_mb": size_mb 
                })
        
        if not candidates:
            print("No mains files found!")
            return

        df_cand = pd.DataFrame(candidates)
        self.all_candidates = df_cand.sort_values("size_mb", ascending=False).to_dict('records')
        print(f"Found {len(self.all_candidates)} total files available for processing.")

    def process_home(self, home_entry):
        home_id = home_entry['homeid']
        path = home_entry['path']
        
        # --- STEP 1: Strict Metadata Check ---
        income = 0
        urban_rural = 0
        tariff = np.nan
        
        if self.home_metadata is not None:
            meta = self.home_metadata[self.home_metadata['homeid'] == home_id]
            if not meta.empty:
                tariff = meta.iloc[0]['unit_charge_pence_per_kwh']
                
                # STRICT FILTER:
                if pd.isna(tariff):
                    return None, "Missing Tariff Data"
                
                if 'income_band' in meta.columns: 
                    income = meta.iloc[0]['income_band']
                if 'urban_rural_class' in meta.columns: 
                    urban_rural = meta.iloc[0]['urban_rural_class']
                    
                
            else:
                return None, "Home not in Metadata"
        else:
             return None, "No Metadata Loaded"
        
        # --- STEP 2: Process Sensor Data ---
        dfs = []
        try:
            for chunk in pd.read_csv(path, header=None, names=["ts", "val"], usecols=[0,1], chunksize=500_000):
                chunk['ts'] = pd.to_datetime(chunk['ts'], utc=True, errors='coerce')
                chunk = chunk.dropna()
                chunk['ts'] = chunk['ts'].dt.floor(self.cfg['FREQ'])
                dfs.append(chunk.groupby('ts')['val'].mean()) 
        except:
            return None, "File Read Error"

        if not dfs: return None, "Empty Data"
        
        full_series = pd.concat(dfs).groupby(level=0).mean()
        
        # Resample & Interpolate (Fixes 5% gaps)
        full_series = full_series.resample(self.cfg['FREQ']).mean()
        full_series = full_series.interpolate(method='linear', limit=4)
        
        e_kwh_series = (full_series * 0.25) / 1000.0
        
        df = pd.DataFrame({'e_kwh': e_kwh_series}).reset_index()
        
        # Assign Features
        df['homeid'] = home_id
        df['unit_charge_pence_per_kwh'] = tariff
        df['income_band'] = income
        df['urban_rural_class'] = urban_rural
        
        # Merge Weather
        if self.weather_data is not None:
            df = df.merge(self.weather_data, on='ts', how='left').ffill()
        else:
            df['temperature'] = 0.0 
            
        # Feature Engineering
        df['tod_sin'] = np.sin(2 * np.pi * (df['ts'].dt.hour * 60 + df['ts'].dt.minute) / 1440)
        df['tod_cos'] = np.cos(2 * np.pi * (df['ts'].dt.hour * 60 + df['ts'].dt.minute) / 1440)
        df['day_of_week_sin'] = np.sin(2 * np.pi * df['ts'].dt.dayofweek / 7)
        df['day_of_week_cos'] = np.cos(2 * np.pi * df['ts'].dt.dayofweek / 7)
        df['e_kwh_lag1'] = df['e_kwh'].shift(1).fillna(0)
        df['e_kwh_roll_mean_4'] = df['e_kwh'].rolling(4).mean().fillna(0)

        # QC: Check Duration
        df = df.dropna(subset=['e_kwh']) 
        
        df['date'] = df['ts'].dt.date
        daily_counts = df.groupby('date')['e_kwh'].count()
        valid_days = daily_counts[daily_counts == 96].index 
        
        if len(valid_days) < self.cfg['REQUIRED_DAYS']:
            return None, f"Too Short (<{self.cfg['REQUIRED_DAYS']} days)"
            
        df = df[df['date'].isin(valid_days)].copy()
        
        # Final Selection
        mandatory_cols = ['ts', 'homeid', 'date']
        final_cols = mandatory_cols + [f for f in self.cfg['FEATURES'] if f in df.columns]
        missing = [f for f in self.cfg['FEATURES'] if f not in df.columns]
        for m in missing:
            df[m] = 0.0
            final_cols.append(m)
        
        return df[final_cols], "Success"

    def run(self):
        self.load_metadata()
        self.load_weather()
        self.find_all_candidates()
        
        valid_agents = []
        target = self.cfg["NUM_AGENTS"]
        print(f"\n--- Searching for {target} Perfect Agents ---")
        
        # Iterate through EVERY file in the dataset
        for i, home in enumerate(self.all_candidates):
            
            # Stop if we hit the target
            if len(valid_agents) >= target:
                print(f"Target reached! Stopping search.")
                break
            
            df, status = self.process_home(home)
            
            if df is not None:
                valid_agents.append(df)
                print(f"[{len(valid_agents)}/{target}] Home {home['homeid']}: ACCEPTED ({len(df)} rows)")
            else:
                # Silent skip, but you can uncomment for debugging
                # print(f"  [x] Home {home['homeid']}: {status}")
                pass

        # --- FINAL SAVE BLOCK ---
        if not valid_agents:
            print("No valid data found! Try reducing REQUIRED_DAYS.")
            return

        # Check if we failed to hit the target count
        if len(valid_agents) < target:
            print(f"\n[WARNING] Scanned all files but only found {len(valid_agents)} agents (Target was {target}).")
            print("Saving partial dataset to prevent data loss...")
        else:
            print(f"\nMerging {len(valid_agents)} agents...")

        panel_df = pd.concat(valid_agents, ignore_index=True)
        out_path = os.path.join(self.cfg["OUTPUT_DIR"], "panel_env_ready_15m.csv.gz")
        panel_df.to_csv(out_path, index=False, compression='gzip')
        print(f"SUCCESS! Saved to {out_path}")
        print(f"Final Count: {panel_df['homeid'].nunique()} Agents.")



In [ ]:
# Run
processor = IdealDataProcessor(CONFIG)
processor.run()